In [2]:
!pip install gradio huggingface_hub sentence-transformers faiss-cpu --quiet

import gradio as gr
from huggingface_hub import InferenceClient
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import csv
import os
import pickle

# Initialize clients
client = InferenceClient(token="your_api_key")  # Your token

# Vector DB setup
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dim = 384

# File paths
FEEDBACK_LOG = "meme_feedback.csv"
VECTOR_DB_FILE = "meme_vector_db.index"
CAPTIONS_FILE = "meme_captions.pkl"

# Initialize/load vector DB
if os.path.exists(VECTOR_DB_FILE) and os.path.exists(CAPTIONS_FILE):
    vector_db = faiss.read_index(VECTOR_DB_FILE)
    with open(CAPTIONS_FILE, 'rb') as f:
        saved_captions = pickle.load(f)
else:
    vector_db = faiss.IndexFlatL2(embedding_dim)
    saved_captions = []

# ===== KEY IMPROVEMENTS =====
def generate_meme(caption):
    # 1. Check duplicates (unchanged)
    similar = find_similar_captions(caption)
    if similar:
        return None, f"⚠ Similar meme exists: '{similar[0]}'"

    # 2. Enhanced generation parameters
    prompt = f"Professional digital art meme: {caption}. Pixar-style, ultra HD, studio lighting"

    image = client.text_to_image(
        prompt=prompt,
        model="stabilityai/stable-diffusion-xl-base-1.0",  # Better than playground-v2.5
        negative_prompt="blurry, distorted, low quality, text, watermark",
        width=1024,
        height=1024,
        guidance_scale=8.5,  # Increased for clarity
        num_inference_steps=35  # More steps = better quality
    )

    # 3. Save to DB (unchanged)
    save_to_vector_db(caption)
    return image, ""

# ===== REST OF YOUR ORIGINAL CODE =====
def get_embedding(text):
    return embedding_model.encode([text])[0]

def find_similar_captions(new_caption, threshold=0.7):
    if len(saved_captions) == 0:
        return []
    new_embedding = get_embedding(new_caption)
    D, I = vector_db.search(np.array([new_embedding]), k=3)
    similar = []
    for distance, idx in zip(D[0], I[0]):
        similarity = 1 / (1 + distance)
        if similarity > threshold:
            similar.append(saved_captions[idx])
    return similar

def save_to_vector_db(caption):
    embedding = get_embedding(caption)
    vector_db.add(np.array([embedding]))
    saved_captions.append(caption)
    faiss.write_index(vector_db, VECTOR_DB_FILE)
    with open(CAPTIONS_FILE, 'wb') as f:
        pickle.dump(saved_captions, f)

def submit_feedback(caption, feedback):
    file_exists = os.path.isfile(FEEDBACK_LOG)
    with open(FEEDBACK_LOG, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(["Caption", "Feedback"])
        writer.writerow([caption, feedback])
    return "✅ Feedback recorded!"

# Gradio Interface (unchanged)
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 Ultra-Clear MemeBot")
    with gr.Row():
        caption_input = gr.Textbox(label="Enter your meme caption", scale=4)
        generate_button = gr.Button("Generate Meme", scale=1)
    image_output = gr.Image(label="Generated Meme")
    similarity_warning = gr.Textbox(label="Similarity Check", visible=True, interactive=False)
    with gr.Row():
        feedback_buttons = gr.Radio(["👍 Funny", "👎 Not Funny"], label="Your Feedback")
        submit_button = gr.Button("Submit Feedback")
    feedback_message = gr.Textbox(visible=False)

    generate_button.click(
        fn=generate_meme,
        inputs=caption_input,
        outputs=[image_output, similarity_warning]
    )
    submit_button.click(
        fn=submit_feedback,
        inputs=[caption_input, feedback_buttons],
        outputs=feedback_message
    )

demo.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30ffd4c4990d294a53.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
